# 04 — SPURS stability scores

SPURS reports raw ΔΔ*G* (wild-type substitutions ≈ 0). Notebook 05 uses
**−SPURS** so that stabilizing substitutions rank higher.

Inference (`get_SPURS_from_hub` + `parse_pdb_resolved_only`) runs **only on
CUDA**. Without a GPU this notebook loads `scores/5lpg_spurs.csv`.

Do not `pip install spurs` from PyPI. See the README for the
`fix/py311-inference-beta` install.


## Resolve scores (GPU scan or cached table)


In [ ]:
from dualde.device import cuda_is_available
from dualde.mapping import load_structural_mapping_json, read_fasta, select_mapping_entry
from dualde.paths import example_dir
from dualde.scores_spurs import add_neg_spurs, load_spurs_table, run_spurs_scan

EXAMPLE = example_dir("NUDT15_HUMAN")
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
CACHE = EXAMPLE / "scores" / "5lpg_spurs.csv"

if cuda_is_available():
    print("CUDA GPU detected; running SPURS.")
    _, uniprot_seq = read_fasta(EXAMPLE / "NUDT15_HUMAN.fasta")
    entry = select_mapping_entry(
        load_structural_mapping_json(EXAMPLE / "mapping" / "5lpg_A.json"),
        pdb_id="5lpg",
        chain="A",
        uniprot_id="NUD15_HUMAN",
    )
    try:
        spurs = run_spurs_scan(
            str(EXAMPLE / "pdb" / "5lpg_cleaned.pdb"),
            entry,
            chain="A",
            uniprot_sequence=uniprot_seq,
            seqres_search_dirs=[str(EXAMPLE / "pdb")],
        )
    except ImportError as err:
        print(err)
        print("Falling back to the precomputed table.")
        spurs = load_spurs_table(CACHE)
else:
    print("No CUDA GPU; loading precomputed SPURS ΔΔG.")
    spurs = load_spurs_table(CACHE)

spurs = add_neg_spurs(spurs)
spurs.to_csv(OUT / "5lpg_spurs.csv", index=False)
print("wrote", OUT / "5lpg_spurs.csv")


## Inspect


In [ ]:
print("shape:", spurs.shape)
print("columns:", list(spurs.columns))
print("UniProt positions:", int(spurs["uniprot_position"].min()), "–", int(spurs["uniprot_position"].max()))
print("n unique positions:", spurs["uniprot_position"].nunique())
spurs.head()


In [ ]:
wt = spurs.loc[spurs["wt"] == spurs["subs"], "spurs"]
print("wild-type diagonal (expect ~0): n =", len(wt), "mean =", float(wt.mean()), "max |ΔΔG| =", float(wt.abs().max()))
spurs[["spurs", "neg_spurs"]].describe().T


In [ ]:
spurs.loc[spurs["uniprot_position"] == 45, ["mutant", "wt", "subs", "spurs", "neg_spurs"]].head(10)
